# Aula 4 — EDA Profissional (Versão Avançada) - Para Aula
**Dataset:** Airbnb NYC

## Objetivos Desta Aula
- **Explorar um dataset real de mercado:** Entender a estrutura e o conteúdo de dados reais.
- **Detectar e tratar valores faltantes e outliers:** Aprender técnicas para limpar e preparar os dados.
- **Realizar análise por categoria e por localização:** Investigar padrões e tendências em diferentes segmentos e regiões.
- **Construir gráficos profissionais e informativos:** Utilizar bibliotecas avançadas para visualizações de alta qualidade.
- **Introduzir correlação entre variáveis:** Entender as relações entre diferentes atributos do dataset.
- **Engenharia de Features:** Criar novas variáveis para enriquecer a análise.
- **Análise Geoespacial:** Visualizar dados em um contexto geográfico.
- **Interpretar e comunicar insights:** Extrair conclusões significativas dos dados.


## 1) Importação das bibliotecas Essenciais
Nesta seção, importamos todas as bibliotecas Python necessárias para a Análise Exploratória de Dados (EDA).

- `pandas`: Para manipulação e análise de dados em formato de DataFrame.
- `matplotlib.pyplot`: Para a criação de gráficos estáticos.
- `seaborn`: Baseado no matplotlib, oferece uma interface de alto nível para criar gráficos estatísticos atraentes e informativos.
- `numpy`: Para operações numéricas e matemáticas, especialmente útil para lidar com arrays e cálculos estatísticos.
- `folium`: Para criar mapas interativos e visualizar dados geoespaciais.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import folium

sns.set_style("whitegrid")  # estilo do seaborn

plt.rcParams['figure.figsize'] = (10, 6)  # tamanho padrão das figuras
plt.rcParams['axes.titlesize'] = 14       # tamanho do título
plt.rcParams['axes.labelsize'] = 12       # tamanho dos rótulos dos eixos
plt.rcParams['xtick.labelsize'] = 10      # tamanho dos ticks do eixo x
plt.rcParams['ytick.labelsize'] = 10   

## 2) Carregar o Dataset
Carregamos o dataset do Airbnb NYC a partir de um arquivo CSV. Este é o primeiro passo para qualquer análise de dados, onde os dados brutos são lidos e transformados em um DataFrame do pandas para fácil manipulação.


In [ ]:
# Carregue o arquivo CSV "AB_NYC_2019.csv" em um DataFrame do pandas
# Exiba as primeiras linhas do DataFrame para uma visão inicial dos dados

df = pd.read_csv("AB_NYC_2019.csv")

df.head(10)

## 3) Visão Geral e Estrutura dos Dados
Esta seção fornece uma visão geral rápida do dataset, incluindo suas dimensões, tipos de dados e um resumo estatístico. Isso nos ajuda a entender a estrutura dos dados e identificar possíveis problemas, como valores ausentes ou tipos de dados incorretos.


In [ ]:
# Exiba o número de linhas e colunas do DataFrame (df.shape)
# Exiba informações sobre o DataFrame, incluindo tipos de dados e valores não-nulos (df.info())
# Exiba estatísticas descritivas para as colunas numéricas (df.describe())
# Exiba estatísticas descritivas para todas as colunas (incluindo categóricas) (df.describe(include="all"))

df.shape

df.info()

df.describe()

df.describe(include="all")


## 4) Tratamento de Valores Faltantes
Valores faltantes são comuns em datasets reais e podem afetar a qualidade da análise. Nesta seção, identificamos e tratamos esses valores, decidindo entre remover, preencher com um valor específico (média, mediana, moda) ou usar métodos mais avançados.


In [ ]:
# Calcule e exiba a quantidade de valores faltantes por coluna
# Preencha os valores faltantes nas colunas `name` e `host_name` com "unknown"
# Preencha os valores faltantes na coluna `reviews_per_month` com 0
# Converta a coluna `last_review` para datetime e preencha os valores faltantes com uma data de referência (e.g., "2010-01-01")
# Verifique novamente os valores faltantes após o tratamento

df.isnull().sum()

df['name'].fillna("unknown", inplace=True)
df['host_name'].fillna("unknown", inplace=True)

df['reviews_per_month'].fillna(0, inplace=True)

df['last_review'] = pd.to_datetime(df['last_review'], errors='coerce')
df['last_review'].fillna(pd.to_datetime("2010-01-01"), inplace=True)

df.isnull().sum()

## 5) Detecção e Tratamento de Outliers
Outliers são pontos de dados que se desviam significativamente de outras observações. Eles podem distorcer análises estatísticas e modelos de machine learning. Nesta seção, identificamos e tratamos outliers em variáveis numéricas chave, como `price` e `minimum_nights`. Usaremos o método do Intervalo Interquartil (IQR) para uma abordagem robusta.


In [ ]:
# Defina uma função `remove_outliers_iqr` para detectar e remover outliers usando o método IQR
# Aplique a função para tratar outliers nas colunas `price`, `minimum_nights` e `number_of_reviews`
# Exiba as estatísticas descritivas antes e depois do tratamento para cada coluna
# Atualize o DataFrame principal com os dados limpos


def remove_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df_clean = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]
    return df_clean

cols = ['price', 'minimum_nights', 'number_of_reviews']

print("ANTES DO TRATAMENTO:\n")
for col in cols:
    print(f"\nColuna: {col}")
    print(df[col].describe())

df_clean = df.copy()
for col in cols:
    df_clean = remove_outliers_iqr(df_clean, col)

print("\n\nDEPOIS DO TRATAMENTO:\n")
for col in cols:
    print(f"\nColuna: {col}")
    print(df_clean[col].describe())

df = df_clean.copy()

## 6) Análise Exploratória de Dados (EDA) Aprofundada
Com os dados limpos, podemos agora aprofundar nossa análise, explorando as distribuições das variáveis, as relações entre elas e identificando padrões que podem gerar insights valiosos.


### 6.1) Distribuição de Preços
Vamos visualizar a distribuição dos preços dos aluguéis para entender a faixa de valores e identificar a concentração de preços.


In [ ]:
# Crie um histograma da coluna `price` usando `sns.histplot`
# Crie um boxplot da coluna `price` usando `sns.boxplot`
# Adicione títulos e rótulos aos gráficos

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure()
sns.histplot(df['price'], bins=50)
plt.title("Distribuição dos Preços")
plt.xlabel("Preço")
plt.ylabel("Frequência")
plt.show()

plt.figure()
sns.boxplot(x=df['price'])
plt.title("Boxplot dos Preços")
plt.xlabel("Preço")
plt.show()


### 6.2) Análise por Tipo de Quarto (`room_type`)
Entender a distribuição dos tipos de quartos e como eles se relacionam com o preço é crucial para proprietários e hóspedes.


In [ ]:
# Crie um gráfico de contagem dos tipos de quarto usando `sns.countplot`
# Crie um boxplot do preço por tipo de quarto usando `sns.boxplot`
# Adicione títulos e rótulos aos gráficos

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure()
sns.countplot(x='room_type', data=df)
plt.title("Quantidade de Anúncios por Tipo de Quarto")
plt.xlabel("Tipo de Quarto")
plt.ylabel("Contagem")
plt.xticks(rotation=45)
plt.show()

plt.figure()
sns.boxplot(x='room_type', y='price', data=df)
plt.title("Distribuição de Preço por Tipo de Quarto")
plt.xlabel("Tipo de Quarto")
plt.ylabel("Preço")
plt.xticks(rotation=45)
plt.show()


### 6.3) Análise por Grupo de Bairro (`neighbourhood_group`)
A localização é um dos fatores mais importantes no mercado imobiliário. Vamos analisar como os preços e a disponibilidade variam entre os diferentes grupos de bairros de NYC.


In [ ]:
# Crie um gráfico de contagem de anúncios por grupo de bairro usando `sns.countplot`
# Crie um boxplot do preço por grupo de bairro usando `sns.boxplot`
# Adicione títulos e rótulos aos gráficos

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure()
sns.countplot(x='neighbourhood_group', data=df)
plt.title("Quantidade de Anúncios por Grupo de Bairro")
plt.xlabel("Grupo de Bairro")
plt.ylabel("Contagem")
plt.xticks(rotation=45)
plt.show()

plt.figure()
sns.boxplot(x='neighbourhood_group', y='price', data=df)
plt.title("Distribuição de Preços por Grupo de Bairro")
plt.xlabel("Grupo de Bairro")
plt.ylabel("Preço")
plt.xticks(rotation=45)
plt.show()

### 6.4) Top 10 Bairros com Mais Anúncios
Identificar os bairros com maior concentração de anúncios pode indicar áreas de alta demanda ou popularidade.


In [ ]:
# Calcule os 10 bairros com mais anúncios
# Crie um gráfico de barras para visualizar esses bairros usando `sns.barplot`
# Adicione títulos e rótulos aos gráficos, e rotacione os rótulos do eixo X se necessário

import matplotlib.pyplot as plt
import seaborn as sns

top_neighbourhoods = df['neighbourhood'].value_counts().head(10)

top_neighbourhoods_df = top_neighbourhoods.reset_index()
top_neighbourhoods_df.columns = ['neighbourhood', 'count']

plt.figure()
sns.barplot(x='neighbourhood', y='count', data=top_neighbourhoods_df)

plt.title("Top 10 Bairros com Mais Anúncios")
plt.xlabel("Bairro")
plt.ylabel("Quantidade de Anúncios")
plt.xticks(rotation=45)

plt.show()


### 6.5) Análise Geoespacial dos Anúncios
Visualizar os anúncios em um mapa nos permite identificar clusters geográficos e entender a distribuição espacial dos imóveis.


In [ ]:
# Crie um mapa base de Nova York usando `folium.Map`
# Adicione um `MarkerCluster` ao mapa
# Amostre uma parte do DataFrame (e.g., 5000 pontos) para melhor performance na visualização
# Adicione marcadores ao mapa para cada ponto amostrado, incluindo informações no popup
# Salve o mapa em um arquivo HTML (e.g., "airbnb_nyc_map.html")
# (Opcional) Exiba o mapa diretamente no notebook se o ambiente suportar

import folium
from folium.plugins import MarkerCluster


map_nyc = folium.Map(location=[40.7128, -74.0060], zoom_start=11)

marker_cluster = MarkerCluster().add_to(map_nyc)

df_sample = df.sample(n=5000, random_state=42)

for _, row in df_sample.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"""
        Nome: {row['name']}<br>
        Preço: ${row['price']}<br>
        Tipo: {row['room_type']}<br>
        Bairro: {row['neighbourhood']}
        """
    ).add_to(marker_cluster)

map_nyc.save("airbnb_nyc_map.html")

map_nyc


## 7) Engenharia de Features
A engenharia de features envolve a criação de novas variáveis a partir das existentes para melhorar a capacidade preditiva dos modelos ou para obter novos insights. Aqui, criaremos algumas features simples.


### 7.1) `host_experience_days`
Calcularemos a experiência do anfitrião em dias, baseando-nos na data da última avaliação (`last_review`) e uma data de referência (ou a data da primeira avaliação, se disponível). Isso pode indicar a longevidade e experiência do anfitrião na plataforma.


In [ ]:
# Calcule a data mais recente de `last_review`
# Crie a nova feature `host_experience_days` calculando a diferença em dias entre a data mais recente e `last_review`
# Exiba as estatísticas descritivas da nova feature

import pandas as pd

df['last_review'] = pd.to_datetime(df['last_review'])

most_recent_date = df['last_review'].max()

df['last_review'] = df['last_review'].fillna(most_recent_date)

df['host_experience_days'] = (most_recent_date - df['last_review']).dt.days

print(df['host_experience_days'].describe())


### 7.2) `price_per_night_per_review`
Esta feature pode indicar o custo-benefício percebido, relacionando o preço por noite com o número de avaliações. Um valor baixo pode indicar um bom negócio, enquanto um valor alto pode sugerir um preço elevado para poucos reviews.


In [ ]:
# Crie a nova feature `price_per_night_per_review`, dividindo `price` por `number_of_reviews`
# Lembre-se de tratar a divisão por zero (e.g., se `number_of_reviews` for 0, use o próprio `price`)
# Exiba as estatísticas descritivas da nova feature

import pandas as pd
import numpy as np

df['price_per_night_per_review'] = np.where(
    df['number_of_reviews'] == 0,
    df['price'],
    df['price'] / df['number_of_reviews']
)

print(df['price_per_night_per_review'].describe())

## 8) Análise de Correlação
A correlação mede a força e a direção da relação linear entre duas variáveis numéricas. Um mapa de calor (heatmap) da matriz de correlação é uma excelente ferramenta para visualizar essas relações.


In [ ]:
# Selecione apenas as colunas numéricas do DataFrame
# Calcule a matriz de correlação
# Crie um heatmap da matriz de correlação usando `sns.heatmap`
# Adicione anotações e um mapa de cores adequado
# Adicione um título ao gráfico

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df_numeric = df.select_dtypes(include=['number'])

corr_matrix = df_numeric.corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')

plt.title('Matriz de Correlação - Airbnb NYC 2019')

plt.show()